# 02 Item2Vec Baseline: Books + Movies Intersection

## Setup

In [1]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
!git checkout item-2-vec && git pull
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

Already up to date.
/content/recommendation-system
Already on 'item-2-vec'
Your branch is up to date with 'origin/item-2-vec'.
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import numpy as np
import pandas as pd
from gensim.models import Word2Vec

from configs.model_configs import ITEM2VEC_CONFIG
from src.data.build_matrix import add_domain_item_ids
from src.evaluation.cross_domain_eval import evaluate_multi_domain


## Load Prepared Splits

In [3]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [4]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()


,user_id,item_id,domain
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0061713244,books
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0871139634,books
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0385521685,books
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0848732634,books
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0151014981,books


## Data prep for item2vec

In [5]:
def build_item_sequences(interactions_df):
    return (
        interactions_df.sort_values('timestamp')
        .groupby('user_id')['item_id']
        .apply(list)
        .tolist()
    )


sequences = build_item_sequences(train_df)

print(f'Total sequences: {len(sequences)}')
print(f'Example: {sequences[0][:5]}')


Total sequences: 127188
Example: ['books::0061058386', 'books::0441008534', 'books::0441009239', 'movies::B0006B2A2O', 'books::0375826688']


## Word2Vec applied to Multi-Domain reommendation problem

In [6]:
def train_item2vec(sequences):
    return Word2Vec(
        sentences=sequences,
        vector_size=ITEM2VEC_CONFIG['vector_size'],
        window=ITEM2VEC_CONFIG['window'],
        sg=ITEM2VEC_CONFIG['sg'],
        min_count=ITEM2VEC_CONFIG['min_count'],
        workers=ITEM2VEC_CONFIG['workers'],
        seed=ITEM2VEC_CONFIG['seed'],
    )


model = train_item2vec(sequences)

print(f'Vocabulary size: {len(model.wv)}')
print(f'Example vector shape: {model.wv[sequences[0][0]].shape}')


Vocabulary size: 332487
Example vector shape: (128,)


# Recommendations

In [7]:
def domain_items_for_model(trained_model, domains=('books', 'movies')):
    return {
        domain: np.array([
            item for item in trained_model.wv.index_to_key
            if item.startswith(f'{domain}::')
        ])
        for domain in domains
    }


domain_items = domain_items_for_model(model)

print(f"Books in vocab:  {len(domain_items['books']):,}")
print(f"Movies in vocab: {len(domain_items['movies']):,}")


Books in vocab:  206,994
Movies in vocab: 125,493


In [8]:
def user_items_by_train_history(train_interactions_df):
    return (
        train_interactions_df.groupby('user_id')['item_id']
        .apply(list)
        .to_dict()
    )


user_train_items = user_items_by_train_history(train_df)


In [9]:
def make_item2vec_recommender(trained_model, train_items_by_user, domains=('books', 'movies')):
    model_domain_items = domain_items_for_model(trained_model, domains=domains)

    def recommend_for_users(user_ids, target_domain, k=10, batch_size=None):
        if batch_size is None:
            batch_size = ITEM2VEC_CONFIG['batch_size']

        candidates = model_domain_items.get(target_domain, np.array([], dtype=object))
        recommendations = {}
        user_ids = list(user_ids)

        if len(candidates) == 0:
            return {user_id: [] for user_id in user_ids}

        candidate_vectors = trained_model.wv[candidates]
        candidate_to_idx = {item: i for i, item in enumerate(candidates)}

        for batch_start in range(0, len(user_ids), batch_size):
            batch = user_ids[batch_start:batch_start + batch_size]
            user_vectors = []
            valid_user_ids = []
            known_items_list = []

            for user_id in batch:
                user_items = train_items_by_user.get(user_id, [])
                valid_items = [item for item in user_items if item in trained_model.wv]
                if not valid_items:
                    recommendations[user_id] = []
                    continue
                user_vectors.append(np.mean(trained_model.wv[valid_items], axis=0))
                valid_user_ids.append(user_id)
                known_items_list.append(set(user_items))

            if not user_vectors:
                continue

            user_matrix = np.stack(user_vectors)
            all_scores = user_matrix @ candidate_vectors.T

            for i, user_id in enumerate(valid_user_ids):
                scores = all_scores[i].copy()
                known_indices = np.array([
                    candidate_to_idx[item]
                    for item in known_items_list[i]
                    if item in candidate_to_idx
                ], dtype=np.int64)
                if len(known_indices) > 0:
                    scores[known_indices] = -np.inf

                finite_count = np.isfinite(scores).sum()
                if finite_count == 0:
                    recommendations[user_id] = []
                    continue

                top_n = min(k, int(finite_count))
                partition_k = min(top_n - 1, len(scores) - 1)
                top_indices = np.argpartition(-scores, partition_k)[:top_n]
                top_indices = top_indices[np.argsort(-scores[top_indices])]
                recommendations[user_id] = [candidates[j] for j in top_indices]

            if batch_start % 10000 == 0:
                print(f"Processed {batch_start}/{len(user_ids)} users...")

        return recommendations

    return recommend_for_users


recommend_for_users = make_item2vec_recommender(model, user_train_items)


## Evaluate and Save Metrics

In [10]:
def item2vec_result_row(model_name, train_interactions_df, valid_results, test_results):
    row = {
        'model': model_name,
        'vector_size': ITEM2VEC_CONFIG['vector_size'],
        'window': ITEM2VEC_CONFIG['window'],
        'sg': ITEM2VEC_CONFIG['sg'],
        'min_count': ITEM2VEC_CONFIG['min_count'],
        'k': ITEM2VEC_CONFIG['k'],
        'n_users': train_interactions_df['user_id'].nunique(),
        'n_items': train_interactions_df['item_id'].nunique(),
        'n_train_interactions': len(train_interactions_df),
    }

    for split_name, results in [('valid', valid_results), ('test', test_results)]:
        for metric in ['ndcg@10', 'recall@10', 'map@10']:
            values = []
            for domain in ['books', 'movies']:
                value = results.get(domain, {}).get(metric, np.nan)
                row[f'{domain}_{split_name}_{metric}'] = value
                if not pd.isna(value):
                    values.append(value)
            row[f'{split_name}_macro_{metric}'] = round(float(np.mean(values)), 4) if values else np.nan

    return row


In [11]:
K = ITEM2VEC_CONFIG['k']

joint_valid_results, joint_test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=recommend_for_users,
    k=K,
)

joint_result_row = item2vec_result_row(
    'item2vec_books_movies_joint',
    train_df,
    joint_valid_results,
    joint_test_results,
)

joint_result_row


[Evaluation] Scoring BOOKS: 127,188 unique users across splits
Processed 0/127188 users...
Processed 10000/127188 users...
Processed 20000/127188 users...
Processed 30000/127188 users...
Processed 40000/127188 users...
Processed 50000/127188 users...
Processed 60000/127188 users...
Processed 70000/127188 users...
Processed 80000/127188 users...
Processed 90000/127188 users...
Processed 100000/127188 users...
Processed 110000/127188 users...
Processed 120000/127188 users...
  -> VALID Results:
     MAP@10: 0.0008 | NDCG@10: 0.0012 | Recall@10: 0.0024
  -> TEST Results:
     MAP@10: 0.0005 | NDCG@10: 0.0007 | Recall@10: 0.0015
[Evaluation] Scoring MOVIES: 127,188 unique users across splits
Processed 0/127188 users...
Processed 10000/127188 users...
Processed 20000/127188 users...
Processed 30000/127188 users...
Processed 40000/127188 users...
Processed 50000/127188 users...
Processed 60000/127188 users...
Processed 70000/127188 users...
Processed 80000/127188 users...
Processed 90000/127

{'model': 'item2vec_books_movies_joint',
 'vector_size': 128,
 'window': 5,
 'sg': 1,
 'min_count': 3,
 'k': 10,
 'n_users': 127188,
 'n_items': 587064,
 'n_train_interactions': 3750813,
 'books_valid_ndcg@10': 0.0012,
 'movies_valid_ndcg@10': 0.0026,
 'valid_macro_ndcg@10': 0.0019,
 'books_valid_recall@10': 0.0024,
 'movies_valid_recall@10': 0.0048,
 'valid_macro_recall@10': 0.0036,
 'books_valid_map@10': 0.0008,
 'movies_valid_map@10': 0.002,
 'valid_macro_map@10': 0.0014,
 'books_test_ndcg@10': 0.0007,
 'movies_test_ndcg@10': 0.0012,
 'test_macro_ndcg@10': 0.0009,
 'books_test_recall@10': 0.0015,
 'movies_test_recall@10': 0.0022,
 'test_macro_recall@10': 0.0019,
 'books_test_map@10': 0.0005,
 'movies_test_map@10': 0.0009,
 'test_macro_map@10': 0.0007}

## Single-Domain Ablation


In [12]:
books_only_sequences = build_item_sequences(books_train)
books_only_model = train_item2vec(books_only_sequences)
books_only_recommend_for_users = make_item2vec_recommender(
    books_only_model,
    user_items_by_train_history(books_train),
    domains=('books',),
)

books_only_valid_results, books_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'books'],
        'test': test_df[test_df['domain'] == 'books'],
    },
    recommend_func=books_only_recommend_for_users,
    k=K,
)

books_only_result_row = item2vec_result_row(
    'item2vec_books_only',
    books_train,
    books_only_valid_results,
    books_only_test_results,
)

books_only_result_row


[Evaluation] Scoring BOOKS: 127,188 unique users across splits
Processed 0/127188 users...
Processed 10000/127188 users...
Processed 20000/127188 users...
Processed 30000/127188 users...
Processed 40000/127188 users...
Processed 50000/127188 users...
Processed 60000/127188 users...
Processed 70000/127188 users...
Processed 80000/127188 users...
Processed 90000/127188 users...
Processed 100000/127188 users...
Processed 110000/127188 users...
Processed 120000/127188 users...
  -> VALID Results:
     MAP@10: 0.0011 | NDCG@10: 0.0017 | Recall@10: 0.0036
  -> TEST Results:
     MAP@10: 0.0009 | NDCG@10: 0.0013 | Recall@10: 0.0027


{'model': 'item2vec_books_only',
 'vector_size': 128,
 'window': 5,
 'sg': 1,
 'min_count': 3,
 'k': 10,
 'n_users': 127188,
 'n_items': 404630,
 'n_train_interactions': 1863230,
 'books_valid_ndcg@10': 0.0017,
 'movies_valid_ndcg@10': nan,
 'valid_macro_ndcg@10': 0.0017,
 'books_valid_recall@10': 0.0036,
 'movies_valid_recall@10': nan,
 'valid_macro_recall@10': 0.0036,
 'books_valid_map@10': 0.0011,
 'movies_valid_map@10': nan,
 'valid_macro_map@10': 0.0011,
 'books_test_ndcg@10': 0.0013,
 'movies_test_ndcg@10': nan,
 'test_macro_ndcg@10': 0.0013,
 'books_test_recall@10': 0.0027,
 'movies_test_recall@10': nan,
 'test_macro_recall@10': 0.0027,
 'books_test_map@10': 0.0009,
 'movies_test_map@10': nan,
 'test_macro_map@10': 0.0009}

In [13]:
movies_only_sequences = build_item_sequences(movies_train)
movies_only_model = train_item2vec(movies_only_sequences)
movies_only_recommend_for_users = make_item2vec_recommender(
    movies_only_model,
    user_items_by_train_history(movies_train),
    domains=('movies',),
)

movies_only_valid_results, movies_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'movies'],
        'test': test_df[test_df['domain'] == 'movies'],
    },
    recommend_func=movies_only_recommend_for_users,
    k=K,
)

movies_only_result_row = item2vec_result_row(
    'item2vec_movies_only',
    movies_train,
    movies_only_valid_results,
    movies_only_test_results,
)

movies_only_result_row


[Evaluation] Scoring MOVIES: 127,188 unique users across splits
Processed 0/127188 users...
Processed 10000/127188 users...
Processed 20000/127188 users...
Processed 30000/127188 users...
Processed 40000/127188 users...
Processed 50000/127188 users...
Processed 60000/127188 users...
Processed 70000/127188 users...
Processed 80000/127188 users...
Processed 90000/127188 users...
Processed 100000/127188 users...
Processed 110000/127188 users...
Processed 120000/127188 users...
  -> VALID Results:
     MAP@10: 0.0025 | NDCG@10: 0.0036 | Recall@10: 0.0072
  -> TEST Results:
     MAP@10: 0.0010 | NDCG@10: 0.0016 | Recall@10: 0.0036


{'model': 'item2vec_movies_only',
 'vector_size': 128,
 'window': 5,
 'sg': 1,
 'min_count': 3,
 'k': 10,
 'n_users': 127188,
 'n_items': 182434,
 'n_train_interactions': 1887583,
 'books_valid_ndcg@10': nan,
 'movies_valid_ndcg@10': 0.0036,
 'valid_macro_ndcg@10': 0.0036,
 'books_valid_recall@10': nan,
 'movies_valid_recall@10': 0.0072,
 'valid_macro_recall@10': 0.0072,
 'books_valid_map@10': nan,
 'movies_valid_map@10': 0.0025,
 'valid_macro_map@10': 0.0025,
 'books_test_ndcg@10': nan,
 'movies_test_ndcg@10': 0.0016,
 'test_macro_ndcg@10': 0.0016,
 'books_test_recall@10': nan,
 'movies_test_recall@10': 0.0036,
 'test_macro_recall@10': 0.0036,
 'books_test_map@10': nan,
 'movies_test_map@10': 0.001,
 'test_macro_map@10': 0.001}

In [14]:
item2vec_ablation_results = pd.DataFrame([
    books_only_result_row,
    movies_only_result_row,
    joint_result_row,
])

os.makedirs('results', exist_ok=True)
item2vec_ablation_results.to_csv('results/item2vec_ablation_metrics.csv', index=False)
item2vec_ablation_results


,model,vector_size,window,sg,min_count,k,n_users,n_items,n_train_interactions,books_valid_ndcg@10,...,valid_macro_map@10,books_test_ndcg@10,movies_test_ndcg@10,test_macro_ndcg@10,books_test_recall@10,movies_test_recall@10,test_macro_recall@10,books_test_map@10,movies_test_map@10,test_macro_map@10
0,item2vec_books_only,128,5,1,3,10,127188,404630,1863230,0.0017,...,0.0011,0.0013,NaN,0.0013,0.0027,NaN,0.0027,0.0009,NaN,0.0009
1,item2vec_movies_only,128,5,1,3,10,127188,182434,1887583,NaN,...,0.0025,NaN,0.0016,0.0016,NaN,0.0036,0.0036,NaN,0.0010,0.0010
2,item2vec_books_movies_joint,128,5,1,3,10,127188,587064,3750813,0.0012,...,0.0014,0.0007,0.0012,0.0009,0.0015,0.0022,0.0019,0.0005,0.0009,0.0007
